# Variable de Control para Opción Asiática

Sofia Anaya palafox

738594

Ingeniería Financiera

In [4]:
import numpy as np
import scipy.stats as stats

# Parámetros del problema
S0 = 100.0
r = 0.05
sigma = 0.20
K = 100.0
T = 1.0
M = 252  # Número de pasos de tiempo (diario en un año)
N = 100  # Número de rutas



## Fórmula Analítica para la Opción Asiática Geométrica

Necesitamos conocer su valor teórico exacto ($C_{geo}$).

In [2]:
def asian_geometric_call_price(S0, K, r, sigma, T, M):
    # Ajuste de parámetros por la media geométrica
    adj_sigma = sigma * np.sqrt((2 * M + 1) / (6 * (M + 1)))
    adj_mu = (r - 0.5 * sigma**2) * 0.5 + 0.5 * adj_sigma**2

    d1 = (np.log(S0 / K) + (adj_mu + 0.5 * adj_sigma**2) * T) / (adj_sigma * np.sqrt(T))
    d2 = d1 - adj_sigma * np.sqrt(T)

    geometric_value = np.exp(-r * T) * (S0 * np.exp(adj_mu * T) * stats.norm.cdf(d1) - K * stats.norm.cdf(d2))
    return geometric_value

C_geo_exact = asian_geometric_call_price(S0, K, r, sigma, T, M)
print(f"Valor teórico exacto de la Call Asiática Geométrica: {C_geo_exact:.6f}")

Valor teórico exacto de la Call Asiática Geométrica: 5.541758


## Simulación de Monte Carlo con Variable de Control

In [3]:
np.random.seed(42)

dt = T / M
Z = np.random.standard_normal((N, M))

# Simulación de rutas
paths = np.zeros((N, M + 1))
paths[:, 0] = S0
for t in range(1, M + 1):
    paths[:, t] = paths[:, t-1] * np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[:, t-1])

arithmetic_means = np.mean(paths[:, 1:], axis=1)
geometric_means = np.exp(np.mean(np.log(paths[:, 1:]), axis=1))

# Payoffs descontados a valor presente
payoff_arithmetic = np.exp(-r * T) * np.maximum(arithmetic_means - K, 0)
payoff_geometric = np.exp(-r * T) * np.maximum(geometric_means - K, 0)

# Estimación estándar (sin variable de control)
MC_arithmetic_mean = np.mean(payoff_arithmetic)
MC_arithmetic_std = np.std(payoff_arithmetic, ddof=1) / np.sqrt(N)

# Estimación con Variable de Control
cov_xy = np.cov(payoff_arithmetic, payoff_geometric)[0, 1]
var_y = np.var(payoff_geometric, ddof=1)
theta = cov_xy / var_y

payoff_control = payoff_arithmetic - theta * (payoff_geometric - C_geo_exact)

MC_control_mean = np.mean(payoff_control)
MC_control_std = np.std(payoff_control, ddof=1) / np.sqrt(N)

print(f"MTC Estándar (Aritmética): {MC_arithmetic_mean:.6f} +/- {1.96 * MC_arithmetic_std:.6f}")
print(f"MTC con Variable de Control: {MC_control_mean:.6f} +/- {1.96 * MC_control_std:.6f}")
print(f"Reducción de la varianza: {np.var(payoff_arithmetic) / np.var(payoff_control):.2f}x")

MTC Estándar (Aritmética): 5.560445 +/- 1.440874
MTC con Variable de Control: 5.741280 +/- 0.034016
Reducción de la varianza: 1794.25x
